In [ ]:
import arcpy
import numpy as np
import random
import math
import os

# 设置工作空间
arcpy.env.workspace = r"D:\ArcGIS\sunleigangPro\模拟退火"
arcpy.env.overwriteOutput = True

# 输入栅格文件
input_raster = "最终ok.tif"

# 输出点要素类
output_points = "max_best_points.shp"
points = []

# 点的高度
point_height = 500  # 单位：米

# 视域范围（半径）
view_distance = 100  # 单位：米

# 目标可见区域比例
target_coverage = 0.9  # 90%
Imax=100

# 将栅格数据加载到 NumPy 数组中
def raster_to_array(raster):
    raster_array = arcpy.RasterToNumPyArray(raster, nodata_to_value=0)
    return raster_array

# 计算栅格的有效面积
def calculate_raster_area(raster):
    cell_size = float(arcpy.GetRasterProperties_management(raster, "CELLSIZEX").getOutput(0))
    raster_array = raster_to_array(raster)
    valid_cell_count = np.count_nonzero(raster_array)
    raster_area = valid_cell_count * (cell_size ** 2)
    return raster_area

# 获取栅格的有效区域多边形
def get_raster_domain(raster):
    # 将栅格的有效区域转换为多边形
    domain_polygon = os.path.join(arcpy.env.workspace, "raster_domain.shp")
    arcpy.RasterDomain_3d(raster, domain_polygon, "POLYGON")
    return domain_polygon



# 计算视域覆盖率
def calculate_coverage():
    
    # 执行视域分析
    viewshed_result = arcpy.sa.Viewshed2(
        in_raster=input_raster,
        in_observer_features="max_temp_points.shp",
        out_agl_raster=None,
        analysis_type="FREQUENCY",
        vertical_error="0 Meters",
        out_observer_region_relationship_table=None,
        refractivity_coefficient=0.13,
        surface_offset="0 Meters",
        observer_elevation=point_height,
        observer_offset="1 Meters",
        inner_radius=None,
        inner_radius_is_3d="GROUND",
        outer_radius=view_distance,
        outer_radius_is_3d="GROUND",
        horizontal_start_angle=0,
        horizontal_end_angle=360,
        vertical_upper_angle=90,
        vertical_lower_angle=-90,
        analysis_method="ALL_SIGHTLINES",
        analysis_target_device="GPU_THEN_CPU"
    )
    
    # 将结果转换为 NumPy 数组
    viewshed_array = raster_to_array(viewshed_result)
    
    # 计算可见区域面积
    unique_values, counts = np.unique(viewshed_array, return_counts=True)
    cell_size = float(arcpy.GetRasterProperties_management(input_raster, "CELLSIZEX").getOutput(0))
    visible_area = 0
    for value, count in zip(unique_values, counts):
        if value > 0:  # 视域值大于0表示可见区域
            visible_area += count * (cell_size ** 2)
    
    # 计算覆盖率
    coverage = visible_area / raster_area
    print(f"当前覆盖率为: {coverage}")
    return coverage

# 算法
def max_points(distance):
   
    # 删除旧的 temp_points.shp 文件
    max_temp_points = os.path.join(arcpy.env.workspace, "max_temp_points.shp")
    if arcpy.Exists(max_temp_points):
        arcpy.management.Delete(max_temp_points)
    i=1    
    best_coverage = 0
    
    # 循环执行
    while(i<Imax):
        random_seed = random.randint(1, 999999)  # 生成一个1到999999之间的随机整数
        random_generator = f"{random_seed} ACM599"  # 使用 ACM599 算法，但每次种子值不同
        with arcpy.EnvManager(randomGenerator=random_generator):
            arcpy.management.CreateRandomPoints(
                out_path=r"D:\ArcGIS\sunleigangPro\模拟退火",
                out_name="max_temp_points",
                constraining_feature_class="raster_domain",
                constraining_extent="DEFAULT",
                number_of_points_or_field=99999,
                minimum_allowed_distance=f"{distance} Meters",
                create_multipoint_output="POINT",
                multipoint_size=0
            )
        current_coverage = calculate_coverage()
        if current_coverage > best_coverage:
            best_coverage = current_coverage
            # 将 max_temp_points.shp 复制为 max_best_points.shp
            arcpy.management.CopyFeatures("max_temp_points.shp", "max_best_points.shp")
        if best_coverage >= target_coverage:
            break
        i += 1
    return best_coverage

# 动态调整点数以满足目标覆盖率
def optimize_coverage():
    
    distance = 2 * view_distance   #最大距离为视域直径
    best_coverage = 0
    
    while(best_coverage < target_coverage):
        best_coverage = max_points(distance)
        if best_coverage < target_coverage:
            print(f"当前最优覆盖率 {best_coverage * 100}% 未达到目标，尝试距离：{distance-1}")
            distance -= 1  
        else:
            print(f"当前最优覆盖率 {best_coverage * 100}% 达到目标，距离为：{distance}")

    return  best_coverage

def extract_and_add_coordinates(output_points):
    # 读取生成的点文件，提取点坐标
    with arcpy.da.SearchCursor(output_points, ["SHAPE@XY"]) as cursor:
        for row in cursor:
            x, y = row[0]  # 提取点的 X 和 Y 坐标
            if not math.isnan(x) and not math.isnan(y):  # 检查坐标是否有效
                points.append((x, y))  # 将坐标添加到列表中
    
    # 添加 X 和 Y 坐标字段
    arcpy.management.AddField(output_points, "X", "DOUBLE")
    arcpy.management.AddField(output_points, "Y", "DOUBLE")
    
    # 更新点数据，将坐标写入 X 和 Y 字段
    with arcpy.da.UpdateCursor(output_points, ["SHAPE@XY", "X", "Y"]) as cursor:
        for i, row in enumerate(cursor):
            if i < len(points):  # 确保不超出 points 列表范围
                x, y = points[i]
                row[1] = x  # 更新 X 字段
                row[2] = y  # 更新 Y 字段
                cursor.updateRow(row)  # 提交更新
                        
# 主程序
if __name__ == "__main__":
    # 获取栅格的有效面积
    raster_area = calculate_raster_area(input_raster)
    print(f"栅格的有效面积: {raster_area} 平方米")
    
    # 获取栅格的有效区域多边形
    domain_polygon = get_raster_domain(input_raster)
    
    # 运行动态调整点数的优化算法
    best_coverage = optimize_coverage()
    
    extract_and_add_coordinates(output_points)  
    
    # 输出结果
    print(f"最优点的数量: {len(points)}")
    print(f"最优点的位置: {points}")
    print(f"可见区域覆盖率: {best_coverage * 100}%")

    # 将生成的要素类加载到地图中
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    map = aprx.listMaps()[0]  # 获取第一个地图
    map.addDataFromPath(os.path.join(arcpy.env.workspace, output_points))
    
    print("处理完成！")

栅格的有效面积: 572800.0 平方米
当前覆盖率为: 0.45216480446927376
当前覆盖率为: 0.48324022346368717
当前覆盖率为: 0.38076117318435754
当前覆盖率为: 0.48411312849162014
当前覆盖率为: 0.3889664804469274
当前覆盖率为: 0.40764664804469275
当前覆盖率为: 0.42946927374301674
当前覆盖率为: 0.4025837988826816
当前覆盖率为: 0.40799581005586594
当前覆盖率为: 0.4478002793296089
当前覆盖率为: 0.4411662011173184
当前覆盖率为: 0.3743016759776536
当前覆盖率为: 0.4958100558659218
当前覆盖率为: 0.46351256983240224
当前覆盖率为: 0.442213687150838
当前覆盖率为: 0.45652932960893855
当前覆盖率为: 0.47817737430167595
当前覆盖率为: 0.3957751396648045
当前覆盖率为: 0.40782122905027934
当前覆盖率为: 0.47695530726256985
当前覆盖率为: 0.40520251396648044
当前覆盖率为: 0.3704608938547486
当前覆盖率为: 0.4661312849162011
当前覆盖率为: 0.3636522346368715
当前覆盖率为: 0.4512918994413408
当前覆盖率为: 0.420391061452514
当前覆盖率为: 0.37587290502793297
当前覆盖率为: 0.4018854748603352
当前覆盖率为: 0.433659217877095
当前覆盖率为: 0.442213687150838
当前覆盖率为: 0.4544343575418994
当前覆盖率为: 0.4715432960893855
当前覆盖率为: 0.4057262569832402
当前覆盖率为: 0.48062150837988826
当前覆盖率为: 0.46752793296089384
当前覆盖率为: 0.36836592178